# Custom Shell Calculation

Run your own shell script against one or more materials on the Mat3ra platform — the shell twin of
the Custom Python Calculation notebook. The script and any files it needs are uploaded to your
object storage folder, a workflow fetches them onto the compute node alongside the material, and
whatever the script prints comes back as the result. A shell script can invoke any application
installed on the node; the default example runs a Quantum ESPRESSO band structure for silicon with
an uploaded pseudopotential.

<h2 style="color:green">Usage</h2>

1. Set the materials and any data files the script reads in cell 1.2. below.
1. Replace the script in cell 1.3. with your own (or keep the default one).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the
   materials, the script, its data files, compute resources and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then
   select account and project.
1. Create materials: materials are read from the `../uploads` folder — place files there manually or
   run a material creation notebook first. If a material is not found by name, Standata is used as a
   fallback. Each material is then saved to the platform.
1. Create workflow: upload the script and its data files, then assemble a workflow that fetches
   them, fetches the material, and runs the script. Optionally save the workflow to the collection.
1. Configure compute: get list of clusters and create compute configuration with selected cluster,
   queue, and number of processors.
1. Create one job per material from the material, workflow, project and compute configuration.
1. Submit the jobs and monitor the status: submit and wait for completion.
1. Retrieve results: read each job's standard output and display the values it printed.

## How the script receives its inputs

Everything lands in the job's working directory, so the script reads it all by **relative path**:

| File | Written by | Contents |
| --- | --- | --- |
| `material.json` | the workflow | the job's material, as stored on the platform |
| your data files | this notebook | uploaded verbatim from `USER_ASSET_FILES` |

The script runs in a shell where `module` is available, so node-side applications load the same way
they do in a command-line job (e.g. `module add espresso`). The script's standard output is the
result. Print JSON and this notebook renders it as a table.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and job

`USER_ASSET_FILES` names data files the script opens, of any type. Put them in the `../uploads`
folder first — drag them into the JupyterLite file browser — and this notebook uploads them
alongside the script.

In [ ]:
import json
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAMES = ["Silicon"]  # One job is created per material

# 4. Script parameters
USER_ASSET_FILES = []  # data files the script opens, taken from FOLDER, e.g. a pseudopotential

# 5. Workflow parameters
WORKFLOW_SEARCH_TERM = "custom_script.json"  # Search term for Workflows Standata
APPLICATION_NAME = "shell"
MY_WORKFLOW_NAME = "Custom Shell Calculation"
save_to_collection = True

# 6. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 7. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set the script to run

This is the calculation. It runs on the compute node with `material.json` and your data files
beside it, and whatever it prints becomes the result. Replace it with your own.

The default example runs a Quantum ESPRESSO band structure for silicon: it builds the pw.x inputs
from `material.json`, runs an SCF and a bands step, and prints the band edges as JSON. It reads the
pseudopotential from the platform's library; to use your own, put the file in `../uploads`, list it
in `USER_ASSET_FILES`, and set `PSEUDO_DIR` in the script to the working directory. The script is uploaded as a file and fetched onto the node, never inlined into
the workflow, so its contents reach the shell exactly as written.

In [ ]:
USER_SCRIPT = r"""#!/bin/bash
# Runs a Quantum ESPRESSO band structure for the job's material.
#
# The pseudopotential comes from the platform's library. To use your own instead, put it in
# uploads, list it in USER_ASSET_FILES, and point PSEUDO_DIR at the working directory ("./").
PSEUDO_DIR="/export/share/pseudo/si/gga/pbe/gbrv/1.0/us/"
PSEUDO_FILES='{"Si": "si_pbe_gbrv_1.0.upf"}'

module add espresso

# Build the pw.x inputs from the job's material.
export PSEUDO_DIR PSEUDO_FILES
python3 - <<'BUILD_INPUTS_EOF'
import json
import math

material = json.load(open("material.json"))
lattice = material["lattice"]
a, b, c = lattice["a"], lattice["b"], lattice["c"]
alpha, beta, gamma = (math.radians(lattice[key]) for key in ("alpha", "beta", "gamma"))
c_x = c * math.cos(beta)
c_y = c * (math.cos(alpha) - math.cos(beta) * math.cos(gamma)) / math.sin(gamma)
vectors = [
    [a, 0.0, 0.0],
    [b * math.cos(gamma), b * math.sin(gamma), 0.0],
    [c_x, c_y, math.sqrt(max(c**2 - c_x**2 - c_y**2, 0.0))],
]
elements = [element["value"] for element in material["basis"]["elements"]]
coordinates = [point["value"] for point in material["basis"]["coordinates"]]

import os
MASSES = {"Si": 28.0855}
PSEUDOS = json.loads(os.environ["PSEUDO_FILES"])
PSEUDO_DIR = os.environ["PSEUDO_DIR"]

cell = "\n".join(" ".join(f"{value:.9f}" for value in vector) for vector in vectors)
positions = "\n".join(f"{el} " + " ".join(f"{x:.9f}" for x in xyz) for el, xyz in zip(elements, coordinates))
species = "\n".join(f"{el} {MASSES[el]} {PSEUDOS[el]}" for el in sorted(set(elements)))

common = f'''&SYSTEM
    ibrav = 0
    nat = {len(elements)}
    ntyp = {len(set(elements))}
    ecutwfc = 40
    ecutrho = 200
    occupations = 'fixed'
    nbnd = 8
/
&ELECTRONS
/
ATOMIC_SPECIES
{species}
CELL_PARAMETERS angstrom
{cell}
ATOMIC_POSITIONS crystal
{positions}
'''

with open("pw_scf.in", "w") as f:
    f.write(f"&CONTROL\n    calculation = 'scf'\n    pseudo_dir = '{PSEUDO_DIR}'\n    outdir = './outdir'\n/\n")
    f.write(common)
    f.write("K_POINTS automatic\n6 6 6 0 0 0\n")

with open("pw_bands.in", "w") as f:
    f.write(f"&CONTROL\n    calculation = 'bands'\n    pseudo_dir = '{PSEUDO_DIR}'\n    outdir = './outdir'\n/\n")
    f.write(common)
    f.write("K_POINTS crystal_b\n3\n0.5 0.5 0.5 20\n0.0 0.0 0.0 20\n0.5 0.0 0.5 20\n")
BUILD_INPUTS_EOF

# Run the calculation. EXEC_CMD is (conditionally) set by the module.
mpirun -np $PBS_NP $EXEC_CMD pw.x -in pw_scf.in > pw_scf.out
mpirun -np $PBS_NP $EXEC_CMD pw.x -in pw_bands.in > pw_bands.out

# Proof and results: QE's own log names the pseudopotential file it read
# (the filename wraps onto the next line, hence -A 1).
grep -A 1 "read from file" pw_scf.out
grep "highest occupied" pw_scf.out

# Band edges at Gamma from the bands run, as JSON for the results table.
python3 - <<'PARSE_BANDS_EOF'
import json
import re

text = open("pw_bands.out").read()
block = text[text.index("End of band structure calculation"):]
gamma = re.search(r"k = 0\.0000 0\.0000 0\.0000[^\n]*\n\n([\s\S]*?)\n\n", block).group(1)
energies = sorted(float(value) for value in re.findall(r"-?\d+\.\d+", gamma))
print(json.dumps({
    "gamma_homo_ev": energies[3],
    "gamma_lumo_ev": energies[4],
    "gamma_direct_gap_ev": round(energies[4] - energies[3], 4),
}))
PARSE_BANDS_EOF
"""

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API Client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Using account: {selected_account.name} ({ACCOUNT_ID})")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Load materials from local files (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

materials = [
    load_material_from_folder(FOLDER, name) or Material.create(Materials.get_by_name_first_match(name))
    for name in MATERIAL_NAMES
]
visualize([{"material": material, "title": material.name} for material in materials])

### 3.2. Save materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [
    Material.create(get_or_create_material(client, material, ACCOUNT_ID)) for material in materials
]

## 4. Create workflow and set its parameters
### 4.1. Upload the script and its data files

The files go to your account's object storage folder ("Dropbox"), which the compute node reads them
from. Text travels inside the request; anything else — a model checkpoint, an archive, a large file
— is sent to a URL the platform signs for it, straight to storage, so there is no size or format
limit beyond what the browser can hold in memory.

In [ ]:
import os

from mat3ra.notebooks_utils.core.entity.file.api import upload_files

files_to_upload = {"user_script.sh": USER_SCRIPT}
for name in USER_ASSET_FILES:
    if name in files_to_upload:
        raise ValueError(f"Rename '{name}' in USER_ASSET_FILES: this notebook already uploads a file by that name.")
    path = os.path.join(FOLDER, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"'{name}' is listed in USER_ASSET_FILES but is not in {FOLDER}.")
    with open(path, "rb") as file:
        content = file.read()
    try:
        files_to_upload[name] = content.decode("utf-8")  # text travels in the request body
    except UnicodeDecodeError:
        files_to_upload[name] = content  # anything else goes to storage through a signed URL

uploaded_files = upload_files(client, files_to_upload, ACCOUNT_ID)

### 4.2. Create workflow from standard workflows and preview it

The `Custom Shell Script` workflow already carries the unit chain this needs: fetch the uploaded
files, fetch the material, run the script. Two things are filled in per job — the objects to fetch
and the runner that hands your script its inputs.

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.core.entity.file.api import to_object_storage_input
from mat3ra.notebooks_utils.core.entity.workflow.api import CUSTOM_SCRIPT_RUNNER_SH, set_execution_unit_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow_config["name"] = MY_WORKFLOW_NAME
subworkflow = workflow_config["subworkflows"][0]
subworkflow["name"] = MY_WORKFLOW_NAME
units = {unit["name"]: unit for unit in subworkflow["units"]}

units["io-user-files"]["input"] = [to_object_storage_input(file) for file in uploaded_files]

set_execution_unit_input(units["custom_script"], "hello_world.sh", CUSTOM_SCRIPT_RUNNER_SH)

workflow = Workflow.create(workflow_config)
visualize_workflow(workflow)

### 4.3. Save workflow to collection

Saving it makes the workflow reusable: it stays in your collection pointing at the uploaded files,
so it can be run again — from the UI or another notebook — against any material. Re-running this
notebook overwrites the uploaded files in place, so a saved workflow picks up an edited script
without being rebuilt.

In [ ]:
saved_workflow = None
if save_to_collection:
    saved_workflow = Workflow.create(
        client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
    )
    print(f"✅ Workflow saved to collection: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN,
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the jobs with material and workflow configuration
### 6.1. Create one job per material

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

jobs = []
for saved_material in saved_materials:
    job_response = create_job(
        api_client=client,
        materials=[saved_material],
        workflow=saved_workflow or workflow,
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}",
        compute=compute.to_dict(),
    )
    jobs.append(job_response if not isinstance(job_response, list) else job_response[0])

job_ids = [job["_id"] for job in jobs]
print(f"✅ Created {len(job_ids)} jobs: {job_ids}")
display_JSON(jobs[0])

## 7. Submit the jobs and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, job_ids)
print(f"✅ Submitted {len(job_ids)} jobs successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 8. Retrieve results

Each job's standard output is the script's own output. Lines that parse as JSON become table
columns; everything else is shown as printed.

In [ ]:
import pandas as pd

from mat3ra.notebooks_utils.io import read_from_url


# The runner names a unit's stdout after the unit, so this is the execution unit's own output -
# not whatever .out file the script itself may have written next to it.
STDOUT_FILENAME = "custom_script.out"


async def read_job_stdout(job_id):
    """Contents of the execution unit's stdout, or why the job produced none."""
    files = client.jobs.list_files(job_id)
    stdout_file = next((file for file in files if file["key"].rsplit("/", 1)[-1] == STDOUT_FILENAME), None)
    if stdout_file is None:
        job = client.jobs.get(job_id)
        errors = job.get("compute", {}).get("errors", [])
        return f"No output. Job status: {job['status']}. {json.dumps(errors, indent=2)}"
    return await read_from_url(stdout_file["signedUrl"])


results = []
for saved_material, job in zip(saved_materials, jobs):
    stdout = await read_job_stdout(job["_id"])
    print(f"--- {job['name']} ---\n{stdout}")
    for line in stdout.splitlines():
        try:
            results.append({"material": saved_material.name, **json.loads(line)})
        except json.JSONDecodeError:
            continue

pd.DataFrame(results)